In [ ]:
!pip -q install -U open_clip_torch SimpleITK

from google.colab import drive
import os, glob, zipfile
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), 'mount failed'

def find(pattern, what):
    hits = glob.glob(pattern, recursive=True)
    print(f'{what}: {len(hits)} found')
    for h in hits[:5]: print('   ', h)
    return hits

find('/content/drive/MyDrive/**/grid_v5_ckpt012.zip', 'grid zip')
find('/content/drive/MyDrive/**/original_data/images', 'node21 images dir')
find('/content/drive/MyDrive/**/original_data/metadata.csv', 'annotations')

In [ ]:
import shutil
from pathlib import Path
import pandas as pd

SRC   = Path('/content/drive/MyDrive/Algoverse/data/grid_v5b_runs/data')
LOCAL = Path('/content/grid')

if not (LOCAL/'grid_v5.csv').exists():
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    print('copying...')
    shutil.copytree(SRC, LOCAL)

print('local contents:', sorted(p.name for p in LOCAL.iterdir()))
GRID = LOCAL
g = pd.read_csv(GRID/'grid_v5.csv', keep_default_na=False, na_values=[''])
print(f'{len(g)} rows | images {len(list((GRID/"images").glob("*.png")))} | '
      f'masks {len(list((GRID/"masks").glob("*.png")))} | '
      f'backgrounds {len(list((GRID/"backgrounds").glob("*.png")))}')
assert len(g) == len(list((GRID/'images').glob('*.png'))), 'copy incomplete'
print('ready')

In [ ]:
from pathlib import Path
import pandas as pd, glob, os, zipfile

NODE21  = Path('/content/drive/MyDrive/Algoverse/data/node21')
MHA_DIR = NODE21/'images'
ANN_CSV = NODE21/'metadata.csv'

assert MHA_DIR.exists(), f'{MHA_DIR} missing'
assert ANN_CSV.exists(), f'{ANN_CSV} missing — contents of {NODE21}: ' \
                         f'{[p.name for p in NODE21.iterdir()]}'

raw = pd.read_csv(ANN_CSV)
print(f'metadata: {raw.shape}  {raw.columns.tolist()}')
pos = raw[raw.label == 1]
print(f'{len(pos)} nodules across {pos.img_name.nunique()} images')
print(f'{len(list(MHA_DIR.glob("*.mha")))} .mha files')

# the grid zip — check Drive, else upload it to /content
GRID = Path('/content/grid'); GRID.mkdir(exist_ok=True)
EMB  = Path('/content/embeddings'); EMB.mkdir(exist_ok=True)

if not (GRID/'grid_v5.csv').exists():
    cand = (glob.glob('/content/drive/MyDrive/**/grid_v5*.zip', recursive=True)
            + glob.glob('/content/grid_v5*.zip'))
    assert cand, ('grid_v5_ckpt012.zip not found. It is on your Mac in '
                  'AV-research-final-review — drag it into /content and re-run.')
    z = max(cand, key=lambda p: os.path.getsize(p))
    print(f'unpacking {z} ({os.path.getsize(z)/1e6:.0f} MB)')
    with zipfile.ZipFile(z) as f: f.extractall(GRID)

g = pd.read_csv(GRID/'grid_v5.csv', keep_default_na=False, na_values=[''])
print(f'\ngrid: {len(g)} rows, {g.chest.nunique()} chests, '
      f'{len(list((GRID/"images").glob("*.png")))} images, '
      f'{len(list((GRID/"backgrounds").glob("*.png")))} backgrounds')

In [ ]:
from pathlib import Path
import pandas as pd

MHA_DIR = Path('/content/drive/MyDrive/Algoverse/data/node21/images')
ANN_CSV = Path('/content/drive/MyDrive/Algoverse/data/node21/metadata.csv')
GRID    = Path('/content/grid')                    # already copied locally
EMB     = Path('/content/embeddings'); EMB.mkdir(exist_ok=True)

assert MHA_DIR.exists() and ANN_CSV.exists() and (GRID/'grid_v5.csv').exists()

g   = pd.read_csv(GRID/'grid_v5.csv', keep_default_na=False, na_values=[''])
raw = pd.read_csv(ANN_CSV)
pos = raw[raw.label == 1]

print(f'grid: {len(g)} rows, {g.chest.nunique()} chests, '
      f'{len(list((GRID/"images").glob("*.png")))} images, '
      f'{len(list((GRID/"backgrounds").glob("*.png")))} backgrounds')
print(f'annotations: {len(pos)} nodules across {pos.img_name.nunique()} images')
print(f'{len(list(MHA_DIR.glob("*.mha")))} .mha files')

In [ ]:
import numpy as np, cv2, SimpleITK as sitk, hashlib, json
from PIL import Image

SIZE = 512

def load_chest(path, size=SIZE):
    """IDENTICAL to the generation pipeline -- crop square (never squash), clip to the
    1st-99th percentile (not min/max), CLAHE, resize. Real and synthetic must go through
    the same transform or any comparison confounds resolution with lesion type."""
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))).astype(np.float32)
    a = a.squeeze() if a.ndim == 3 else a
    oh, ow = a.shape
    s = min(oh, ow); x0, y0 = (ow-s)//2, (oh-s)//2
    a = a[y0:y0+s, x0:x0+s]
    lo, hi = np.percentile(a, [1, 99])
    a = np.clip((a-lo)/(hi-lo+1e-8), 0, 1)
    a = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)) \
           .apply((a*255).astype(np.uint8)).astype(np.float32)/255.0
    a = np.clip(cv2.resize(a, (size,size), interpolation=cv2.INTER_AREA), 0, 1)
    return Image.fromarray((a*255).astype(np.uint8)).convert('RGB'), (ow, oh, s, x0, y0)

In [ ]:
import torch, open_clip

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
clip_model.eval().to(DEV)
print('BiomedCLIP on', DEV)

@torch.no_grad()
def embed(pil):
    return clip_model.encode_image(
        preprocess(pil.convert('RGB')).unsqueeze(0).to(DEV)).cpu().numpy().ravel()

def lesion_crop(pil, box, pad=1.5):
    """Square crop centred on the lesion, pad x the box width.

    The whole-image embedding is dominated by the chest: cluster membership tracked source
    identity at AMI 0.326 while cluster-failure was only 0.143. Embedding a crop as well
    turns that suspicion into a measurement -- if the crop predicts and the full image does
    not, the original 0.885 was reading which radiograph it was, not whether it failed.
    """
    W, H = pil.size
    cx, cy = (box[0]+box[2])/2*W, (box[1]+box[3])/2*H
    r = max(box[2]-box[0], box[3]-box[1]) * W * pad / 2
    return pil.crop((max(0, cx-r), max(0, cy-r), min(W, cx+r), min(H, cy+r)))

In [ ]:
full, crop = {}, {}
for i, r in g.iterrows():
    im = Image.open(GRID/'images'/f'{r.image_id}.png')
    full[r.image_id] = embed(im)
    crop[r.image_id] = embed(lesion_crop(im, (r.mask_x0, r.mask_y0, r.mask_x1, r.mask_y1)))
    if i % 100 == 0: print(f'  {i}/{len(g)}')

np.savez_compressed(EMB/'synth_full.npz', **full)
np.savez_compressed(EMB/'synth_crop.npz', **crop)
print(f'synthetic: {len(full)} full + {len(crop)} crop, dim {len(next(iter(full.values())))}')

In [ ]:
bg = {p.stem: embed(Image.open(p)) for p in sorted((GRID/'backgrounds').glob('*.png'))}
np.savez_compressed(EMB/'background_full.npz', **bg)
print(f'backgrounds: {len(bg)}')
# these answer a different question: BEFORE generating, will RadEdit paint on this chest?
# That is the predictor with a balanced target (419 painted / 205 not).

In [ ]:
rfull, rcrop, meta = {}, {}, []
for i, (name, grp) in enumerate(pos.groupby('img_name')):
    p = MHA_DIR/(name if str(name).endswith('.mha') else f'{name}.mha')
    if not p.exists(): continue
    img, (W, H, s, x0, y0) = load_chest(p)
    stem = str(name).replace('.mha', '')
    rfull[stem] = embed(img)

    for j, r in enumerate(grp.itertuples()):
        fx0, fy0 = (r.x-x0)/s, (r.y-y0)/s
        fx1, fy1 = (r.x+r.width-x0)/s, (r.y+r.height-y0)/s
        if not (0 <= fx0 < fx1 <= 1 and 0 <= fy0 < fy1 <= 1):
            continue                      # drop, never clamp -- this was F0
        key = f'{stem}__{j}'
        rcrop[key] = embed(lesion_crop(img, (fx0, fy0, fx1, fy1)))
        meta.append(dict(key=key, img_name=stem, fx0=round(fx0,4), fy0=round(fy0,4),
                         fx1=round(fx1,4), fy1=round(fy1,4)))
    if i % 100 == 0: print(f'  {i} images, {len(rcrop)} nodule crops')

np.savez_compressed(EMB/'real_full.npz', **rfull)
np.savez_compressed(EMB/'real_crop.npz', **rcrop)
pd.DataFrame(meta).to_csv(EMB/'real_crop_index.csv', index=False)
print(f'real: {len(rfull)} images, {len(rcrop)} nodule crops')

In [ ]:
ok = True
for f in sorted(EMB.glob('*.npz')):
    z = np.load(f)
    keys = list(z.keys())
    X = np.stack([z[k] for k in keys])
    u    = len(np.unique(X, axis=0))
    zero = int((np.abs(X).sum(1) == 0).sum())
    nan  = int(np.isnan(X).any(1).sum())
    print(f'{f.name:<24} {str(X.shape):<12} unique {u}/{len(X)}'
          f'{"  <-- DUPLICATES" if u != len(X) else ""}'
          f'{f"  {zero} all-zero" if zero else ""}'
          f'{f"  {nan} with NaN" if nan else ""}')
    ok &= (u == len(X)) and zero == 0 and nan == 0

_id = list(np.load(EMB/'synth_full.npz').keys())[0]
a = embed(Image.open(GRID/'images'/f'{_id}.png'))
b = np.load(EMB/'synth_full.npz')[_id]
print(f'\nreproducible: max deviation {np.abs(a-b).max():.2e}')
ok &= np.abs(a-b).max() < 1e-4

h = {f.name: hashlib.md5(f.read_bytes()).hexdigest() for f in sorted(EMB.glob('*.npz'))}
json.dump(h, open(EMB/'embeddings_hashes.json', 'w'), indent=2)

assert ok, 'validation failed -- do not use these'
print('\nvalidated. A previous feature file held one vector repeated 180 times (F2) and')
print('nothing caught it for weeks. That is what the unique-row check is for.')

In [ ]:
import numpy as np
z = np.load(EMB/'real_full.npz')
keys = list(z.keys()); X = np.stack([z[k] for k in keys])

_, inv, cnt = np.unique(X, axis=0, return_inverse=True, return_counts=True)
for grp in np.where(cnt > 1)[0]:
    dupes = [keys[i] for i in np.where(inv == grp)[0]]
    print('identical embeddings:', dupes)

    # are the source images actually identical?
    import hashlib
    for d in dupes:
        p = MHA_DIR/f'{d}.mha'
        print(f'  {d}.mha  {p.stat().st_size/1e6:.2f} MB  '
              f'md5 {hashlib.md5(p.read_bytes()).hexdigest()[:12]}')
    a, _ = load_chest(MHA_DIR/f'{dupes[0]}.mha')
    b, _ = load_chest(MHA_DIR/f'{dupes[1]}.mha')
    diff = np.abs(np.asarray(a.convert('L'), np.float32)
                  - np.asarray(b.convert('L'), np.float32))
    print(f'  pixel difference after preprocessing: mean {diff.mean():.3f}, '
          f'max {diff.max():.1f}')

    print('\n  annotations:')
    print(raw[raw.img_name.isin([f'{d}.mha' for d in dupes])].to_string(index=False))

In [ ]:
KNOWN_DUPES = {...}          # fill in from the output above
ok = True
for f in sorted(EMB.glob('*.npz')):
    z = np.load(f); keys = list(z.keys()); X = np.stack([z[k] for k in keys])
    u = len(np.unique(X, axis=0))
    allowed = len(KNOWN_DUPES) if f.name == 'real_full.npz' else 0
    bad = (len(X) - u) > allowed
    print(f'{f.name:<24} {str(X.shape):<12} unique {u}/{len(X)}'
          f'{"  <-- UNEXPECTED DUPLICATES" if bad else ""}')
    ok &= not bad
assert ok, 'validation failed'

In [ ]:
DROP = 'n0507'          # identical image to n1059; n1059 has the more complete annotation
print(f'dropping {DROP}: {(pos.img_name == f"{DROP}.mha").sum()} nodules, '
      f'{len(pos)} -> {len(pos) - (pos.img_name == f"{DROP}.mha").sum()}')
pos = pos[pos.img_name != f'{DROP}.mha']

z = dict(np.load(EMB/'real_full.npz'))
z.pop(DROP, None)
np.savez_compressed(EMB/'real_full.npz', **z)

zc = {k: v for k, v in np.load(EMB/'real_crop.npz').items() if not k.startswith(DROP)}
np.savez_compressed(EMB/'real_crop.npz', **zc)
print(f'real_full {len(z)}, real_crop {len(zc)}')

In [ ]:
import shutil
DEST = Path('/content/drive/MyDrive/Algoverse/embeddings')
assert os.path.ismount('/content/drive'), 'Drive not mounted'
DEST.mkdir(parents=True, exist_ok=True)
for f in EMB.iterdir():
    shutil.copy(f, DEST/f.name)
print(f'{len(list(EMB.iterdir()))} files -> {DEST}  [safe]')

z = shutil.make_archive('/content/embeddings', 'zip', EMB)
print(f'{Path(z).stat().st_size/1e6:.1f} MB')
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print(f'download skipped ({e}) -- Drive copy is intact, re-run this cell')